# Kaggle — trening CarRacing (SAC + LightCNN)

## Setup (once per notebook)

1. **Settings → Accelerator → GPU T4 x1**
2. **Add Data** → code dataset (upload **new version** after `python scripts/package_kaggle_bundle.py`)
3. **Run All** → download `kaggle_outputs.zip` from **Output**

## Modes

| MODE | When | What |
|------|------|------|
| `long_run` | done | single long baseline seed0 |
| **`sweep`** | **now** | 3 HP × seeds, skip-existing, max 3 runs/session |

**Overrides v2:** `frame_stack: 4`, eval every 25k — needs **repackaged dataset** + new training (old stack=2 models still work locally for demo).

## After each session (local)

```bash
python scripts/import_kaggle_outputs.py --zip ~/Downloads/kaggle_outputs.zip
python scripts/plot_curves.py --arch arch_light_cnn --min-timesteps 100000
python scripts/watch_agent.py --run-dir experiments/hp_baseline__arch_light_cnn__seed02__20260519-074316 --fast --episodes 5
```

In [ ]:
"""Locate repo root from the attached Kaggle dataset."""
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
EXPERIMENTS = WORK / "experiments"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)

print("=== sample files under /kaggle/input ===")
if INPUT.is_dir():
    shown = 0
    for p in sorted(INPUT.rglob("*")):
        if not p.is_file():
            continue
        rel = p.relative_to(INPUT)
        if len(rel.parts) <= 5:
            print(rel)
            shown += 1
        if shown >= 30:
            print("... (truncated)")
            break
else:
    print("EMPTY — Add Data → twój dataset z kodem")

CODE_ROOT = None
for candidate in sorted(INPUT.rglob("run_experiment.py")):
    if candidate.parent.name == "scripts":
        CODE_ROOT = candidate.parent.parent
        break

if CODE_ROOT is None:
    raise FileNotFoundError(
        "Brak scripts/run_experiment.py w /kaggle/input.\n"
        "Uruchom lokalnie: python scripts/package_kaggle_bundle.py\n"
        "Wrzuć dist/racing-agent-kaggle.zip jako Kaggle Dataset i Add Data."
    )

print("CODE_ROOT:", CODE_ROOT)
print("EXPERIMENTS:", EXPERIMENTS)
print("run_experiment:", CODE_ROOT / "scripts" / "run_experiment.py")

In [ ]:
"""Install Box2D + project deps (Kaggle GPU image)."""
import subprocess
import sys
from pathlib import Path

if Path("/kaggle").is_dir():
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "swig"], check=False)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "swig", "Box2D==2.3.10", "pygame>=2.1.0"],
    check=True,
)
req = CODE_ROOT / "requirements-kaggle.txt"
if not req.is_file():
    raise FileNotFoundError(f"Missing {req}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

import Box2D  # noqa: F401
import gymnasium as gym  # noqa: F401
import torch

print("Box2D + gymnasium OK")
print("CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" —", torch.cuda.get_device_name(0))
else:
    print(" (WARNING: włącz GPU T4 w Settings!)")

In [ ]:
import sys

# === MODE: use "sweep" now ===
MODE = "sweep"
SWEEP_TIMESTEPS = "100000"   # 50000 for faster sessions; 100000 ~42 min/run on T4

RUN_SCRIPT = CODE_ROOT / "scripts" / "run_experiment.py"
COMMON = [
    sys.executable,
    str(RUN_SCRIPT),
    "--arch",
    "arch_light_cnn",
    "--overrides",
    "kaggle_overrides",
    "--output-root",
    str(EXPERIMENTS),
    "--skip-existing",
    "--max-wall-clock-s",
    "28000",
]

if MODE == "long_run":
    RUN_CMD = COMMON + [
        "--configs", "hp_baseline",
        "--seeds", "0",
        "--timesteps", "100000",
        "--max-runs", "1",
    ]
elif MODE == "sweep":
    RUN_CMD = COMMON + [
        "--configs", "hp_baseline", "hp_high_lr", "hp_large_batch",
        "--seeds", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
        "--timesteps", SWEEP_TIMESTEPS,
        "--max-runs", "3",
    ]
else:
    raise ValueError(f"Unknown MODE={MODE!r}")

print("MODE:", MODE)
print("Command:\n ", " ".join(RUN_CMD))

In [ ]:
"""Train (writes to /kaggle/working/experiments/)."""
import subprocess

print("Starting training…")
proc = subprocess.run(RUN_CMD, cwd=str(CODE_ROOT))
if proc.returncode != 0:
    raise RuntimeError(f"run_experiment.py failed with exit code {proc.returncode}")

runs = sorted(p.name for p in EXPERIMENTS.iterdir() if p.is_dir())
print(f"Done. {len(runs)} run folder(s) under {EXPERIMENTS}")
for name in runs[-5:]:
    print(" ", name)
if len(runs) > 5:
    print("  ...")

In [ ]:
"""Zip experiments/ for download."""
import json
import subprocess

export_script = CODE_ROOT / "scripts" / "export_kaggle_outputs.py"
out_zip = WORK / "kaggle_outputs.zip"

subprocess.run(
    [
        sys.executable,
        str(export_script),
        "--experiments-dir",
        str(EXPERIMENTS),
        "--output",
        str(out_zip),
    ],
    cwd=str(CODE_ROOT),
    check=True,
)

print("Download from Output tab:", out_zip)
print(json.dumps({"mode": MODE, "zip": str(out_zip), "runs": len(list(EXPERIMENTS.iterdir()))}, indent=2))